In [ ]:
# =============================================================================
#  HEAVENLY REBELLION — MAX QUALITY Cloud Video Pipeline
#  Just run this cell, then run Cell 2. That's it!
# =============================================================================

import os, sys, shutil, subprocess
from pathlib import Path

# ── Step 1: Mount Google Drive (remembers you automatically) ─────────────────
from google.colab import drive
import shutil

def mount_drive_robustly():
    mountpoint = '/content/drive'
    mydrive = '/content/drive/MyDrive'
    
    # 1. Check if already mounted and working
    if os.path.exists(mydrive) and os.path.isdir(mydrive):
        try:
            # Try to list directory contents to verify it is responsive
            os.listdir(mydrive)
            print('[+] Google Drive is already mounted and working perfectly.')
            return
        except Exception as e:
            print(f'[*] Active mount point detected but unresponsive: {e}. Re-mounting...')
            
    # 2. If the mountpoint exists and is not empty, unmount or rename it
    if os.path.exists(mountpoint) and os.path.isdir(mountpoint) and os.listdir(mountpoint):
        print('[!] Mountpoint exists and is not empty. Clean unmounting...')
        try:
            drive.flush_and_unmount()
            print('[+] Clean unmount successful.')
        except Exception as e:
            print(f'[-] Clean unmount bypassed: {e}')
            
        # Try force unmounting using system commands
        subprocess.run(['umount', '-f', mountpoint], capture_output=True)
        subprocess.run(['fusermount', '-u', mountpoint], capture_output=True)
        import time
        time.sleep(2)
        
        # If it still contains files, rename it to get it out of the way safely
        if os.path.exists(mountpoint) and os.path.isdir(mountpoint) and os.listdir(mountpoint):
            backup_name = f"/content/drive_backup_{int(time.time())}"
            print(f'[!] Mountpoint still contains files. Renaming {mountpoint} to {backup_name}...')
            try:
                os.rename(mountpoint, backup_name)
                print('[+] Renamed successfully.')
            except Exception as e:
                print(f'[!] Rename failed: {e}. Trying standard mount anyway...')
                
    # 3. Mount Google Drive cleanly
    print('[*] Mounting Google Drive to /content/drive ...')
    try:
        drive.mount(mountpoint)
        print('[+] Drive mounted successfully!')
    except Exception as e:
        print(f'[!] Standard mount failed: {e}. Trying force remount...')
        try:
            drive.mount(mountpoint, force_remount=True)
            print('[+] Force remount successful!')
        except Exception as e2:
            print(f'[x] Mount failed completely: {e2}')
            raise e2

try:
    mount_drive_robustly()
except Exception as e:
    print(f'[x] Mount failed: {e}')

# Force-refresh Google Drive directory cache in Colab to avoid FileNotFoundError on newly uploaded files
print('[*] Refreshing Google Drive directory cache...')
try:
    os.listdir('/content/drive')
    if os.path.exists('/content/drive/MyDrive'):
        os.listdir('/content/drive/MyDrive')
        if os.path.exists('/content/drive/MyDrive/HeavenlyRebellion'):
            os.listdir('/content/drive/MyDrive/HeavenlyRebellion')
    print('[+] Cache refresh complete.')
except Exception as e:
    print(f'[!] Warning during cache refresh: {e}')

DRIVE_FOLDER = Path('/content/drive/MyDrive/HeavenlyRebellion')
assert DRIVE_FOLDER.exists(), f'ERROR: {DRIVE_FOLDER} not found. Run colab_launcher.py on your PC first!'
print(f'[+] Drive folder found: {DRIVE_FOLDER}')

# ── Step 2: Install packages (skips if already installed) ────────────────────
def pkg_installed(name):
    try:
        __import__(name)
        return True
    except ImportError:
        return False

deps = []
if not pkg_installed('edge_tts'):
    deps.append('edge-tts')
if not pkg_installed('nest_asyncio'):
    deps.append('nest_asyncio')

if deps:
    print(f'[*] Installing missing packages: {", ".join(deps)} ...')
    subprocess.run(['pip', 'install'] + deps + ['-q'], check=True)
else:
    print('[+] All dependencies (edge-tts, nest_asyncio) are already installed.')

# Install ffmpeg if missing
ffmpeg_check = subprocess.run(['which', 'ffmpeg'], capture_output=True)
if ffmpeg_check.returncode != 0:
    print('[*] Installing FFmpeg ...')
    subprocess.run('apt-get update -qq && apt-get install -y ffmpeg -qq', shell=True, check=True)
else:
    print('[+] FFmpeg already installed — skipping.')

# ── Step 3: GPU check ────────────────────────────────────────────────────────
gpu = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
if gpu.returncode == 0:
    print(f'[+] GPU: {gpu.stdout.strip()}')
else:
    print('[!] No GPU — CPU mode (slower but still works)')

# ── Step 4: Copy latest pipeline scripts from Drive to Colab SSD ─────────────
shutil.copy(DRIVE_FOLDER / 'colab_pipeline.py',  '/content/colab_pipeline.py')
shutil.copy(DRIVE_FOLDER / 'story_expander.py',  '/content/story_expander.py')
sys.path.insert(0, '/content')
sys.path.insert(0, str(DRIVE_FOLDER))

# Verify it is the MAX QUALITY version
content = Path('/content/colab_pipeline.py').read_text()
if 'NVENC_CQ' in content and 'NVENC_PRESET' in content:
    print('[+] MAX QUALITY pipeline confirmed (CQ18, 1920x1080, h264_nvenc p7, 30fps)')
else:
    print('[!] WARNING: Old pipeline version detected. Re-run colab_launcher.py on your PC.')

# ── Step 5: Verify all required files ────────────────────────────────────────
files = sorted(DRIVE_FOLDER.iterdir())
print(f'\n[+] Files in Drive ({len(files)} total):')
for f in files:
    print(f'    {f.name:50s}  {f.stat().st_size/1024:8.1f} KB')

print('\n[+] SETUP COMPLETE — Run Cell 2 to start the pipeline!')


In [ ]:
# =============================================================================
#  RUN THE FULL PIPELINE
#  Estimated time: 2-4 hours (images take longest)
#  Your PowerShell window on your laptop will auto-download when done.
# =============================================================================

import runpy
runpy.run_path('/content/colab_pipeline.py', run_name='__main__')
